# Test Case 2 — v31 vs v32 Causality Engine Comparison

## Purpose

This notebook runs a **full integration test** of the RCA orchestrator pipeline on a realistic
condenser vacuum loss scenario. It compares the outputs of two causality engine versions side by
side to validate that both agree on the primary root cause while demonstrating the effect of
the stricter evidence-threshold filtering introduced in v32.

Unlike Test Case 1 (which supplies all intermediate artifacts as fixtures), this test exercises
the **live evidence retrieval path**: processed plant documents are ingested into a local Chroma
vector store and queried by the orchestrator's evidence retriever during the run.

## Scenario

| Field | Value |
|---|---|
| **Event ID** | defined in `fixtures/event.json` |
| **Asset** | Main condenser (secondary side of a PWR unit) |
| **Primary root cause** | `FM::FM_AIR_INLEAK` — air in-leakage degrading condenser vacuum |
| **Competing hypotheses** | Tube fouling, vacuum instrumentation bias, feedwater control valve instability, HVAC support degradation |
| **Telemetry signals** | 5 signals: condenser vacuum (`COND_VAC_A`), turbine back-pressure (`TBP_A`), hotwell level (`HOTWELL_LVL_A`), feedwater flow (`FW_FLOW_A`), air ejector DP (`AIR_EJECTOR_DP_A`) |
| **Evidence corpus** | 5 processed plant documents (CRs, SOPs, inspection records) in `processed_records.jsonl` |

## Engine versions under test

| Version | Description | Candidate filtering |
|---|---|---|
| **v31** | Baseline engine — structural + evidence scoring only | None (all generated candidates are retained) |
| **v32** | Production engine — adds TSKR Allen-relation temporal scoring and `EntityNormalizer` NER enrichment | Evidence/governance threshold filtering applied after scoring |

Both engines share the same fixture inputs and the same evidence store. Any difference in
candidates or scores is attributable solely to the engine logic, not to input variation.
This makes the notebook suitable as a **cross-validation and ablation tool**.

## What this test covers

- Live evidence retrieval via `ProcessedEvidenceStoreAdapter` → `LCProcessedRetriever` → ChromaDB
- `ChromaRecordStore` ingestion from a JSONL file of processed plant documents
- Parallel orchestrator runs for v31 and v32 against identical inputs
- Candidate ranking and `composite_score` comparison across engines
- Candidate filtering behavior in v32 (`generated_candidate_count` vs `retained_candidate_count`)
- Ishikawa matrix population from candidates, evidence snippets, and telemetry
- Full artifact persistence: both runs saved to `rca_runs_case_002/{v31,v32}_full_result.json`

## What this test does NOT cover

- **Live Neo4j queries** — `kg_context` is pre-supplied; the Neo4j client is instantiated but
  not queried for KG neighborhood expansion
- **LLM synthesis** — the fallback rule-based synthesizer is used (`fallback_used: true`);
  Ollama is not required
- **TSKR live scoring** — `tskr_patterns.json` is pre-supplied when present; the TSKR scorer
  only runs if that file is absent from the fixture directory

## Files required

| Path | Required | Description |
|---|---|---|
| `fixtures/event.json` | Yes | Abnormal event descriptor |
| `fixtures/telemetry_summary.json` | Yes | Aggregated sensor anomaly summary |
| `fixtures/kg_context.json` | Yes | Pre-built KG context (components, failure modes, past events, documents) |
| `fixtures/processed_records.jsonl` | Yes | Processed plant documents for Chroma ingestion |
| `fixtures/tskr_patterns.json` | No | Pre-built TSKR temporal patterns; the TSKR scorer runs live if absent |
| `fixtures/operational_context.json` | No | Operational state context |
| `fixtures/pm_compliance.json` | No | Preventive maintenance compliance record |

Chroma data is persisted under `chroma_case_002/` in the notebook directory.
Re-running the notebook will re-ingest the JSONL (upsert is idempotent).

## Expected outputs

| Field | v31 | v32 |
|---|---|---|
| `primary_candidate_id` | `FM::FM_AIR_INLEAK` | `FM::FM_AIR_INLEAK` |
| `n_candidates` (retained) | 5 | 2 |
| `generated_candidate_count` | — | 5 |
| `filtered_out_candidate_count` | — | 3 |
| `fallback_used` | `true` | `true` |
| `decision_status` | `review_required` | `review_required` |

The agreement on `FM::FM_AIR_INLEAK` as the primary hypothesis across both engines,
despite v32 discarding three lower-confidence candidates, validates that the additional
temporal scoring in v32 does not reverse the ranking of the top candidate.

## Pipeline architecture reference

```
Stage A — Build run context
Stage B — KG context builder     ← pre-supplied from fixture
Stage C — TSKR temporal scorer   ← pre-supplied if tskr_patterns.json exists
Stage D — Causality engine       ← runs live (v31 or v32)
Stage E — Evidence retriever     ← runs live (Chroma vector store)
Stage F — RCA synthesizer        ← runs live (fallback rule-based)
Stage G — Finalize manifest      ← runs live (validation + writeback check)
```

Stages D–G run with live logic. Stage B is bypassed. Stage C is bypassed when
`tskr_patterns.json` is present in the fixture directory.


In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path
from typing import Any, Dict, Optional

In [ ]:
NOTEBOOK_ROOT = Path.cwd().resolve()

FIXTURE_DIR = NOTEBOOK_ROOT / "fixtures" 
OUTPUT_DIR = NOTEBOOK_ROOT / "rca_runs_case_002"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------
# Python path setup
# ---------------------------------------------------------------------
# Update these if your notebook is in a different location.
rca_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if rca_root not in sys.path:
    sys.path.insert(0, rca_root)

dackar_root = os.path.abspath(os.path.join(os.getcwd(), "..", "..", ".."))
if dackar_root not in sys.path:
    sys.path.insert(0, dackar_root)

# ---------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------
from orchestrators.rca_reasoning_orchestrator import build_dev_orchestrator
from kg.py2neo_workflow import Py2Neo
from storage.chroma_store import ChromaRecordStore
from storage.processed_record_store import ProcessedRecordStore
from storage.lc_retriever_processed import LCProcessedRetriever
from storage.processed_evidence_store_adapter import ProcessedEvidenceStoreAdapter

# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "password")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE", None)

VALIDATOR_MODE = "compat"
STOP_ON_VALIDATION_ERROR = False

# Robust schema path from installed/imported package location
import orchestrators.rca_reasoning_orchestrator as orch_mod
SCHEMA_DIR = Path(orch_mod.__file__).resolve().parents[1] / "schemas"

print("Fixture dir:", FIXTURE_DIR)
print("Schema dir :", SCHEMA_DIR)
print("Schemas    :", sorted(p.name for p in SCHEMA_DIR.glob("*.json")))

## Utility functions

- `load_json(path)` — load a required JSON file; raises `FileNotFoundError` if absent
- `maybe_load_json(path)` — load an optional JSON file; returns `None` if absent
- `safe_get(d, *keys, default)` — safe nested dict lookup across arbitrary depth
- `require_file(path)` — assert that a required fixture file exists before loading
- `print_block(title, obj, max_chars)` — pretty-print a JSON-serializable object with
  a character limit to avoid flooding the output

In [ ]:
def load_json(path: Path) -> Dict[str, Any]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def maybe_load_json(path: Path) -> Optional[Dict[str, Any]]:
    return load_json(path) if path.exists() else None

def safe_get(d: Optional[Dict[str, Any]], *keys: str, default=None):
    cur = d
    for k in keys:
        if not isinstance(cur, dict):
            return default
        cur = cur.get(k)
    return default if cur is None else cur

def require_file(path: Path) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing required fixture file: {path}")

def print_block(title: str, obj: Any, max_chars: int = 5000) -> None:
    print(f"\n--- {title} ---")
    text = json.dumps(obj, indent=2, default=str)
    print(text[:max_chars])

## Load fixture bundle

Loads the three required fixture files (`event`, `telemetry_summary`, `kg_context`) and
any optional pre-built artifacts. The `require_file` guard raises `FileNotFoundError`
immediately if a required file is missing, rather than propagating a confusing error later.

- `tskr_patterns.json` — if present, the TSKR temporal scorer is bypassed; the orchestrator
  uses this pre-computed result directly
- `operational_context.json` — plant operating state (mode, load level, active alarms)
- `pm_compliance.json` — preventive maintenance schedule compliance for the affected asset

In [ ]:
required_files = [
    "event.json",
    "telemetry_summary.json",
    "kg_context.json",
]

for name in required_files:
    require_file(FIXTURE_DIR / name)

event = load_json(FIXTURE_DIR / "event.json")
telemetry_summary = load_json(FIXTURE_DIR / "telemetry_summary.json")
kg_context = load_json(FIXTURE_DIR / "kg_context.json")

# Optional prebuilt artifacts
tskr_patterns = maybe_load_json(FIXTURE_DIR / "tskr_patterns.json")
operational_context = maybe_load_json(FIXTURE_DIR / "operational_context.json")
pm_compliance = maybe_load_json(FIXTURE_DIR / "pm_compliance.json")

## Fixture sanity checks

Asserts that `event_id` and `asset_id` are consistent across all loaded artifacts.
These checks catch fixture mismatches (e.g., a `kg_context` from a different event)
before any pipeline logic runs.

The schema directory check below verifies that `causality_candidates.json` is present
in the runtime schema directory, confirming that the validator will be able to validate
the causality candidates artifact produced by Stage D.

In [ ]:
assert event["event_id"] == telemetry_summary["event_id"], "event_id mismatch"
assert event["asset_id"] == telemetry_summary["asset_id"], "asset_id mismatch"
assert kg_context["event_id"] == event["event_id"], "kg_context.event_id mismatch"
assert kg_context["asset_id"] == event["asset_id"], "kg_context.asset_id mismatch"

if tskr_patterns is not None:
    assert tskr_patterns["event_id"] == event["event_id"], "tskr_patterns.event_id mismatch"
    assert tskr_patterns["asset_id"] == event["asset_id"], "tskr_patterns.asset_id mismatch"

print("Fixture sanity checks passed.")


In [ ]:
print("Schema dir :", SCHEMA_DIR)
print("Schemas    :", sorted(p.name for p in SCHEMA_DIR.glob("*.json")))
assert (SCHEMA_DIR / "causality_candidates.json").exists(), "Missing causality_candidates.json in runtime schema dir"

## Build the evidence store

Constructs the vector store pipeline that the orchestrator's evidence retriever will query
during Stage E. The stack has three layers:

1. **`ProcessedRecordStore`** — in-memory index of all processed plant documents, keyed
   by `record_id`. Loaded from `processed_records.jsonl`.

2. **`ChromaRecordStore`** — persistent Chroma vector store. Documents are upserted from
   the JSONL file. The `upsert_jsonl` call is idempotent; re-running the notebook will
   not create duplicates. Embeddings are generated by Ollama (`nomic-embed-text` by
   default) if Ollama is reachable, or fall back to a null embedder.

3. **`LCProcessedRetriever`** — LangChain-compatible retriever that combines dense
   (Chroma ANN) and sparse (BM25) retrieval with Reciprocal Rank Fusion (RRF).

4. **`ProcessedEvidenceStoreAdapter`** — wraps the retriever in the interface expected
   by `EvidenceRetriever`, applying per-doc-type top-K limits and RRF fusion.

> **Note:** A LangChain deprecation warning about `OllamaEmbeddings` is expected and
> harmless. It will be resolved when the dependency is updated to `langchain-ollama`.

In [ ]:
# Chroma adapter for v32
# ## Build evidence store

# %%
PROCESSED_JSONL = FIXTURE_DIR / "processed_records.jsonl"  # adjust if needed
CHROMA_DIR = NOTEBOOK_ROOT / "chroma_case_002"

record_store = ProcessedRecordStore([str(PROCESSED_JSONL)])
chroma_manager = ChromaRecordStore(
    persist_directory=str(CHROMA_DIR),
)

# Ingest once for the local test corpus
chroma_manager.upsert_jsonl(str(PROCESSED_JSONL))

lc_retriever = LCProcessedRetriever(
    manager=chroma_manager,
    doc_store=record_store,
)

evidence_store = ProcessedEvidenceStoreAdapter(
    retriever=lc_retriever,
    k_final=10,
)

print("evidence_store:", evidence_store)
print("record count:", len(record_store))

## Build the orchestrators

Creates two orchestrator instances — one for v31 and one for v32 — wired to the
same evidence store, Neo4j client, schema directory, and validator.

The only difference between the two instances is the `causality_engine_version`
argument passed to `build_dev_orchestrator`:

| Parameter | v31 | v32 |
|---|---|---|
| Causality engine | `CausalityEngineV31` | `CausalityEngineV32` |
| TSKR Allen-relation scoring | No | Yes |
| `EntityNormalizer` NER enrichment | No | Yes |
| Candidate filtering threshold | None | Evidence/governance threshold |

The debug cell that follows confirms that the evidence store is correctly wired into
the v32 orchestrator's `evidence_retriever` attribute.

In [ ]:
client = Py2Neo(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD)

orchestrator_v31 = build_dev_orchestrator(
    output_dir=OUTPUT_DIR / "v31",
    client=client,
    database=NEO4J_DATABASE,
    evidence_store=evidence_store,
    schema_dir=SCHEMA_DIR,
    validator_mode=VALIDATOR_MODE,
    stop_on_validation_error=STOP_ON_VALIDATION_ERROR,
    causality_engine_version="v31",
)

orchestrator_v32 = build_dev_orchestrator(
    output_dir=OUTPUT_DIR / "v32",
    client=client,
    database=NEO4J_DATABASE,
    evidence_store=evidence_store,
    schema_dir=SCHEMA_DIR,
    validator_mode=VALIDATOR_MODE,
    stop_on_validation_error=STOP_ON_VALIDATION_ERROR,
    causality_engine_version="v32",
)

print("Orchestrators built.")
print("Validator schemas:", sorted(getattr(orchestrator_v31.validator, "schemas", {}).keys()))

In [ ]:
print("\n=== DEBUG: Evidence store wiring ===")
print("orchestrator_v32.evidence_retriever.store:", orchestrator_v32.evidence_retriever.store)
print("type:", type(orchestrator_v32.evidence_retriever.store))

# Optional deeper inspection
store = orchestrator_v32.evidence_retriever.store
if hasattr(store, "__dict__"):
    print("store attributes:", list(store.__dict__.keys()))

## Run both orchestrators

Executes both pipeline runs sequentially against the same input bundle.
Each run produces an independent `result` dict with the same top-level keys.

Both runs share a single Neo4j client connection, which is closed in the `finally`
block regardless of success or failure. If a run raises an exception, the partially
completed result will not be available for inspection.

Because `kg_context` is pre-supplied, neither run issues Neo4j queries. Stage D
(causality engine) and Stage E (evidence retrieval) run with live logic.

In [ ]:
try:
    result_v31 = orchestrator_v31.run(
        event=event,
        telemetry_summary=telemetry_summary,
        operational_context=operational_context,
        pm_compliance=pm_compliance,
        kg_context=kg_context,
        tskr_patterns=tskr_patterns,
    )
    result_v32 = orchestrator_v32.run(
        event=event,
        telemetry_summary=telemetry_summary,
        operational_context=operational_context,
        pm_compliance=pm_compliance,
        kg_context=kg_context,
        tskr_patterns=tskr_patterns,
    )

finally:
    client.close()

print("Runs completed.")

## Inspect and compare outputs

Builds a concise comparison summary for each run and prints it side by side.

Key fields to compare:

| Field | What it tells you |
|---|---|
| `primary_candidate_id` | Both engines must agree on the top hypothesis |
| `n_candidates` | v31 retains all generated candidates; v32 filters by threshold |
| `generated_candidate_count` / `retained_candidate_count` | v32 only — shows how many candidates were filtered |
| `fallback_used` | `true` means the rule-based synthesizer was used (no LLM call) |
| `writeback_ready` | `false` means the result requires analyst review before persisting |
| `decision_status` | `review_required` — standard outcome without LLM confirmation |

In [ ]:
def build_comparison_summary(label: str, result: Dict[str, Any]) -> Dict[str, Any]:
    return {
        "label": label,
        "run_id": safe_get(result, "run_context", "run_id"),
        "engine_version": safe_get(result, "run_manifest", "pipeline_config", "causality_engine_version"),
        "decision_status": safe_get(result, "rca_card", "executive_summary", "decision_status"),
        "primary_candidate_id": safe_get(result, "rca_card", "primary_hypothesis", "candidate_id"),
        "n_candidates": len(safe_get(result, "causality_candidates", "candidates", default=[]) or []),
        "n_evidence": len(safe_get(result, "evidence_bundle", "results", default=[]) or []),
        "supporting_count": safe_get(result, "run_manifest", "artifacts", "rca_card", "primary_supporting_evidence_count"),
        "contradicting_count": safe_get(result, "run_manifest", "artifacts", "rca_card", "primary_contradicting_evidence_count"),
        "fallback_used": safe_get(result, "rca_card", "validation_status", "fallback_used"),
        "writeback_ready": safe_get(result, "run_manifest", "review_hooks", "writeback_ready"),
        "generated_candidate_count": safe_get(result, "causality_candidates", "summary", "generated_candidate_count"),
        "retained_candidate_count": safe_get(result, "causality_candidates", "summary", "retained_candidate_count"),
        "filtered_out_candidate_count": safe_get(result, "causality_candidates", "summary", "filtered_out_candidate_count"),
    }

summary_v31 = build_comparison_summary("v31", result_v31)
summary_v32 = build_comparison_summary("v32", result_v32)

print_block("summary_v31", summary_v31, max_chars=3000)
print_block("summary_v32", summary_v32, max_chars=3000)

## Candidate ranking and Ishikawa matrix

Sorts each engine's candidates by `composite_score` (descending) and prints the
ranking alongside the Allen temporal relation and matched telemetry signal IDs.

**Expected v31 ranking:**
1. `FM::FM_AIR_INLEAK` — score ≈ 0.776, temporal relation `precedes`
2. `FM::FM_HVAC_SUPPORT_DEGRAD` — score ≈ 0.736, temporal relation `precedes`
3. `FM::FM_COND_FOULING` — score ≈ 0.735, temporal relation `simultaneous`
4. `FM::FM_VAC_INST_BIAS` — score ≈ 0.720, temporal relation `unknown`
5. `FM::FM_FWCV_INSTAB` — score ≈ 0.686, temporal relation `follows`

**Expected v32 ranking (post-filtering):**
1. `FM::FM_AIR_INLEAK` — score ≈ 0.742, temporal relation `precedes`
2. `FM::FM_HVAC_SUPPORT_DEGRAD` — score ≈ 0.662, temporal relation `precedes`

The Ishikawa matrix organizes all evidence across five standard cause categories:
`equipment_hardware`, `process_procedure`, `measurement_instrumentation`,
`environment_operating_context`, and `maintenance_human_factors`.
Each row shows the contributing factor, its strength score, and the source artifact
(causality candidates, evidence bundle, telemetry, or KG context).

In [ ]:
for label, result in [("v31", result_v31), ("v32", result_v32)]:
    cands = safe_get(result, "causality_candidates", "candidates", default=[]) or []
    cands_sorted = sorted(
        cands,
        key=lambda c: float(c.get("composite_score", 0.0)),
        reverse=True,
    )
    print(f"\n=== Candidate ranking: {label} ===")
    for i, c in enumerate(cands_sorted, start=1):
        print(
            f"{i}. {c.get('candidate_id')} | "
            f"{c.get('cause_label')} | "
            f"score={c.get('composite_score')} | "
            f"temporal={safe_get(c, 'temporal_evidence', 'relation')} | "
            f"signals={safe_get(c, 'telemetry_evidence', 'matching_signal_ids', default=[])}"
        )

    # Ishikawa summary
    cats = safe_get(result, "ishikawa_matrix", "categories", default=[]) or []
    print(f"\n=== Ishikawa categories: {label} ===")
    for cat in cats:
        category = cat.get("category")
        rows = cat.get("rows", []) or []
        print(f"{category}: {len(rows)} rows")
        for row in rows[:3]:
            print(
                "   -",
                row.get("label"),
                "| strength=",
                row.get("strength"),
                "| source=",
                row.get("source_artifact"),
            )

## Save full result bundles

Writes the complete result dict for each engine version to disk:
- `rca_runs_case_002/v31_full_result.json`
- `rca_runs_case_002/v32_full_result.json`

These files contain all pipeline artifacts (kg_context, tskr_patterns,
causality_candidates, evidence_bundle, ishikawa_matrix, rca_card, run_manifest)
and can be used as fixtures for downstream tests or as audit records.

In [ ]:
for label, result in [("v31", result_v31), ("v32", result_v32)]:
    out_path = OUTPUT_DIR / f"{label}_full_result.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2, default=str)
    print(f"Saved {label} full result to:", out_path)